In [93]:
import pandas as pd
import numpy as np
import os
from glob import glob
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

In [94]:
OUTPUT_DIR = "./processed_data/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [95]:
# OBD-II 标准特征列名
OBD_FEATURES = [
    'ENGINE_RUN_TINE ()', 'ENGINE_RPM ()', 'VEHICLE_SPEED ()', 'THROTTLE ()', 'ENGINE_LOAD ()',
    'COOLANT_TEMPERATURE ()', 'LONG_TERM_FUEL_TRIM_BANK_1 ()', 'SHORT_TERM_FUEL_TRIM_BANK_1 ()',
    'INTAKE_MANIFOLD_PRESSURE ()', 'FUEL_TANK ()', 'ABSOLUTE_THROTTLE_B ()',
    'PEDAL_D ()', 'PEDAL_E ()', 'COMMANDED_THROTTLE_ACTUATOR ()',
    'FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'ABSOLUTE_BAROMETRIC_PRESSURE ()', 'RELATIVE_THROTTLE_POSITION ()',
    'INTAKE_AIR_TEMP ()', 'TIMING_ADVANCE ()', 'CATALYST_TEMPERATURE_BANK1_SENSOR1 ()',
    'CATALYST_TEMPERATURE_BANK1_SENSOR2 ()', 'CONTROL_MODULE_VOLTAGE ()', 'COMMANDED_EVAPORATIVE_PURGE ()',
    'TIME_RUN_WITH_MIL_ON ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()', 'DISTANCE_TRAVELED_WITH_MIL_ON ()',
    'WARM_UPS_SINCE_CODES_CLEARED ()'
]

# 滑窗参数
WINDOW_SIZE = 60  # 60秒窗口
STRIDE = 10       # 步长10秒

In [96]:
def load_and_merge_data():
    """读取所有CSV，添加场景标签，并合并"""
    all_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    df_list = []
    
    for file_path in all_files:
        # 从文件名提取场景（如 idle, drive, live等）
        file_name = os.path.basename(file_path)
        scenario = file_name.split('.')[0]  # 例如 'idle1', 'drive2'
        
        # 读取CSV，假设第一行为列名
        try:
            df = pd.read_csv(file_path)
            # 统一列名：去掉首尾空格
            df.columns = df.columns.str.strip()
            # 只保留我们定义的OBD特征列（忽略可能存在的其他列）
            existing_features = [col for col in OBD_FEATURES if col in df.columns]
            if not existing_features:
                print(f"警告: 文件 {file_name} 中未找到匹配的OBD特征列，跳过。")
                continue
            df = df[existing_features].copy()
            # 添加场景标签
            df['SCENARIO'] = scenario
            df_list.append(df)
        except Exception as e:
            print(f"读取文件 {file_name} 时出错: {e}")
    
    if not df_list:
        raise ValueError("未成功读取任何数据文件，请检查数据路径和文件格式。")
    
    # 合并所有数据
    merged_df = pd.concat(df_list, ignore_index=True)
    print(f"成功合并数据，总行数: {len(merged_df)}")
    return merged_df

In [97]:
def clean_and_convert_to_numeric(df, feature_cols):
    """清洗并转换数据为数值类型"""
    df_clean = df.copy()
    
    for col in feature_cols:
        # 1. 去除空格并替换空字符串为NaN
        if df_clean[col].dtype == 'object':
            df_clean[col] = df_clean[col].astype(str).str.strip()
            df_clean[col] = df_clean[col].replace(['', ' ', 'NaN', 'nan', 'NULL', 'null'], np.nan)
        
        # 2. 转换为数值类型，无法转换的变为NaN
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean

In [98]:
def handle_missing_and_outliers(df, feature_cols):
    """处理缺失值和异常值"""
    # 1. 先清洗并转换数据类型
    df_clean = clean_and_convert_to_numeric(df, feature_cols)
    
    # 2. 缺失值处理：先向前填充，再向后填充
    df_clean[feature_cols] = df_clean[feature_cols].ffill().bfill()
    
    # 3. 检查是否还有缺失值（如果某列全部为NaN，填充为0）
    for col in feature_cols:
        if df_clean[col].isna().all():
            print(f"警告: 列 {col} 全部为缺失值，填充为0")
            df_clean[col] = 0
        elif df_clean[col].isna().any():
            # 剩余少量缺失值用中位数填充
            median_val = df_clean[col].median()
            df_clean[col] = df_clean[col].fillna(median_val)
            print(f"列 {col} 有少量缺失值，用中位数 {median_val} 填充")
    
    # 4. 基于业务规则的异常值处理
    if 'ENGINE_RPM' in feature_cols:
        df_clean['ENGINE_RPM'] = df_clean['ENGINE_RPM'].clip(lower=0, upper=8000)
    
    if 'COOLANT_TEMPERATURE' in feature_cols:
        df_clean['COOLANT_TEMPERATURE'] = df_clean['COOLANT_TEMPERATURE'].clip(lower=-40, upper=150)
    
    if 'CONTROL_MODULE_VOLTAGE' in feature_cols:
        df_clean['CONTROL_MODULE_VOLTAGE'] = df_clean['CONTROL_MODULE_VOLTAGE'].clip(lower=6, upper=16)
    
    if 'VEHICLE_SPEED' in feature_cols:
        df_clean['VEHICLE_SPEED'] = df_clean['VEHICLE_SPEED'].clip(lower=0, upper=250)
    
    return df_clean

In [99]:
def normalize_features(df, feature_cols):
    """对数值特征进行标准化 (Z-score)，增加错误处理"""
    try:
        # 确保所有特征都是数值类型
        numeric_data = df[feature_cols].values
        
        # 检查是否有非数值数据
        if not np.issubdtype(numeric_data.dtype, np.number):
            # 尝试再次转换
            df_temp = df[feature_cols].apply(pd.to_numeric, errors='coerce')
            numeric_data = df_temp.values
        
        # 检查是否存在NaN或Inf
        if np.isnan(numeric_data).any() or np.isinf(numeric_data).any():
            print("警告: 数据包含NaN或Inf值，将用0替换")
            numeric_data = np.nan_to_num(numeric_data, nan=0.0, posinf=0.0, neginf=0.0)
        
        # 创建标准化器并拟合
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(numeric_data)
        
        # 创建新的DataFrame
        df_scaled = df.copy()
        df_scaled[feature_cols] = scaled_data
        
        print(f"标准化成功，特征数: {len(feature_cols)}")
        return df_scaled, scaler
        
    except Exception as e:
        print(f"标准化过程出错: {e}")
        # 如果标准化失败，使用MinMaxScaler作为备选
        print("尝试使用MinMaxScaler作为备选方案...")
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
        numeric_data = df[feature_cols].values
        numeric_data = np.nan_to_num(numeric_data, nan=0.0)
        scaled_data = scaler.fit_transform(numeric_data)
        df_scaled = df.copy()
        df_scaled[feature_cols] = scaled_data
        return df_scaled, scaler

In [100]:
def create_sliding_windows(data, window_size, stride):
    """将时间序列数据转换为滑窗样本 (N, window_size, F)"""
    num_samples = (len(data) - window_size) // stride + 1
    
    # 检查是否有足够的样本
    if num_samples <= 0:
        raise ValueError(f"数据长度({len(data)})小于窗口大小({window_size})，无法创建滑窗")
    
    windows = []
    for i in range(num_samples):
        start_idx = i * stride
        end_idx = start_idx + window_size
        window = data[start_idx:end_idx]
        windows.append(window)
    
    return np.array(windows)

In [101]:
def validate_data_integrity(data, feature_cols):
    """验证数据完整性"""
    print("\n=== 数据验证 ===")
    print(f"数据形状: {data.shape}")
    print(f"特征数量: {len(feature_cols)}")
    
    # 检查是否有NaN
    if np.isnan(data).any():
        nan_count = np.isnan(data).sum()
        print(f"警告: 数据中包含 {nan_count} 个NaN值")
        print("将NaN替换为0...")
        data = np.nan_to_num(data, nan=0.0)
    
    # 检查是否有Inf
    if np.isinf(data).any():
        inf_count = np.isinf(data).sum()
        print(f"警告: 数据中包含 {inf_count} 个Inf值")
        print("将Inf替换为0...")
        data = np.nan_to_num(data, posinf=0.0, neginf=0.0)
    
    # 检查数值范围
    print(f"数据范围: min={data.min():.4f}, max={data.max():.4f}")
    print(f"数据均值: {data.mean():.4f}")
    print(f"数据标准差: {data.std():.4f}")
    
    return data

In [102]:
# ================= 主执行流程 =================
if __name__ == "__main__":
    try:
        print("=== 步骤1: 读取并合并数据 ===")
        raw_df = load_and_merge_data()
        
        # 确定可用的特征列
        feature_cols = [col for col in OBD_FEATURES if col in raw_df.columns]
        print(f"\n使用的特征列 ({len(feature_cols)}个): {feature_cols}")
        
        print("\n=== 步骤2: 处理缺失值和异常值 ===")
        cleaned_df = handle_missing_and_outliers(raw_df, feature_cols)
        
        # 保存场景标签供后续分析
        scenario_labels = cleaned_df['SCENARIO'].values
        print(f"场景类型: {np.unique(scenario_labels)}")
        
        print("\n=== 步骤3: 标准化特征 ===")
        normalized_df, scaler = normalize_features(cleaned_df, feature_cols)
        
        # 提取数值数据
        numeric_data = normalized_df[feature_cols].values
        
        # 验证数据完整性
        numeric_data = validate_data_integrity(numeric_data, feature_cols)
        
        print(f"\n=== 步骤4: 创建滑窗数据 (窗口={WINDOW_SIZE}, 步长={STRIDE}) ===")
        windowed_data = create_sliding_windows(numeric_data, WINDOW_SIZE, STRIDE)
        
        # 生成窗口的场景标签（取窗口内出现最多的场景）
        windowed_scenarios = []
        for i in range(windowed_data.shape[0]):
            start_idx = i * STRIDE
            end_idx = start_idx + WINDOW_SIZE
            window_scenarios = scenario_labels[start_idx:end_idx]
            most_common = pd.Series(window_scenarios).mode()[0]
            windowed_scenarios.append(most_common)
        
        print(f"\n=== 保存预处理数据 ===") 
        # 保存数据
        np.save(os.path.join(OUTPUT_DIR, "windowed_data.npy"), windowed_data)
        np.save(os.path.join(OUTPUT_DIR, "windowed_scenarios.npy"), np.array(windowed_scenarios))
        
        # 保存标准化器
        import joblib
        joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.pkl"))
        
        # 保存特征列名
        import json
        with open(os.path.join(OUTPUT_DIR, "feature_columns.json"), 'w') as f:
            json.dump(feature_cols, f)
        
        print(f"✅ 预处理完成！")
        print(f"   - 数据形状: {windowed_data.shape}")
        print(f"   - 样本数量: {windowed_data.shape[0]}")
        print(f"   - 窗口长度: {windowed_data.shape[1]} 秒")
        print(f"   - 特征数量: {windowed_data.shape[2]}")
        print(f"   - 保存位置: {OUTPUT_DIR}")
        
        # 显示一些统计信息  
        print(f"\n=== 场景分布 ===")
        scenario_counts = pd.Series(windowed_scenarios).value_counts()
        for scenario, count in scenario_counts.items():
            print(f"   {scenario}: {count} ({count/len(windowed_scenarios)*100:.1f}%)")
            
    except Exception as e:
        print(f"\n❌ 预处理过程出现错误: {e}")
        import traceback
        traceback.print_exc()

=== 步骤1: 读取并合并数据 ===
成功合并数据，总行数: 304299

使用的特征列 (27个): ['ENGINE_RUN_TINE ()', 'ENGINE_RPM ()', 'VEHICLE_SPEED ()', 'THROTTLE ()', 'ENGINE_LOAD ()', 'COOLANT_TEMPERATURE ()', 'LONG_TERM_FUEL_TRIM_BANK_1 ()', 'SHORT_TERM_FUEL_TRIM_BANK_1 ()', 'INTAKE_MANIFOLD_PRESSURE ()', 'FUEL_TANK ()', 'ABSOLUTE_THROTTLE_B ()', 'PEDAL_D ()', 'PEDAL_E ()', 'COMMANDED_THROTTLE_ACTUATOR ()', 'FUEL_AIR_COMMANDED_EQUIV_RATIO ()', 'ABSOLUTE_BAROMETRIC_PRESSURE ()', 'RELATIVE_THROTTLE_POSITION ()', 'INTAKE_AIR_TEMP ()', 'TIMING_ADVANCE ()', 'CATALYST_TEMPERATURE_BANK1_SENSOR1 ()', 'CATALYST_TEMPERATURE_BANK1_SENSOR2 ()', 'CONTROL_MODULE_VOLTAGE ()', 'COMMANDED_EVAPORATIVE_PURGE ()', 'TIME_RUN_WITH_MIL_ON ()', 'TIME_SINCE_TROUBLE_CODES_CLEARED ()', 'DISTANCE_TRAVELED_WITH_MIL_ON ()', 'WARM_UPS_SINCE_CODES_CLEARED ()']

=== 步骤2: 处理缺失值和异常值 ===
场景类型: ['drive1' 'drive10' 'drive11' 'drive12' 'drive13' 'drive2' 'drive3'
 'drive4' 'drive5' 'drive6' 'drive7' 'drive8' 'drive9' 'idle1' 'idle10'
 'idle11' 'idle12' 'idle